In [3]:
from os.path import join

import nflreadpy as nfl
import pathlib
import polars as pl

In [7]:
# set up to load player data for seasons
seasons_for_class = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

available_seasons = [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014,
                     2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [10]:
# get player data for exploratory analysis
player_stats_2022 = nfl.load_player_stats(
    seasons=seasons_for_class,
    summary_level="reg"
)

In [11]:
# exploratory analysis of player data
# print(type(player_stats_2022))

# col headers

# print(player_stats_2022.columns)
col_headers = player_stats_2022.columns
# ['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'season', 'season_type', 'recent_team', 'games', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'pacr', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share', 'wopr', 'special_teams_tds', 'def_tackles_solo', 'def_tackles_with_assist', 'def_tackle_assists', 'def_tackles_for_loss', 'def_tackles_for_loss_yards', 'def_fumbles_forced', 'def_sacks', 'def_sack_yards', 'def_qb_hits', 'def_interceptions', 'def_interception_yards', 'def_pass_defended', 'def_tds', 'def_fumbles', 'def_safeties', 'misc_yards', 'fumble_recovery_own', 'fumble_recovery_yards_own', 'fumble_recovery_opp', 'fumble_recovery_yards_opp', 'fumble_recovery_tds', 'penalties', 'penalty_yards', 'punt_returns', 'punt_return_yards', 'kickoff_returns', 'kickoff_return_yards', 'fg_made', 'fg_att', 'fg_missed', 'fg_blocked', 'fg_long', 'fg_pct', 'fg_made_0_19', 'fg_made_20_29', 'fg_made_30_39', 'fg_made_40_49', 'fg_made_50_59', 'fg_made_60_', 'fg_missed_0_19', 'fg_missed_20_29', 'fg_missed_30_39', 'fg_missed_40_49', 'fg_missed_50_59', 'fg_missed_60_', 'fg_made_list', 'fg_missed_list', 'fg_blocked_list', 'fg_made_distance', 'fg_missed_distance', 'fg_blocked_distance', 'pat_made', 'pat_att', 'pat_missed', 'pat_blocked', 'pat_pct', 'gwfg_made', 'gwfg_att', 'gwfg_missed', 'gwfg_blocked', 'gwfg_distance_list', 'fantasy_points', 'fantasy_points_ppr']

# col datatypes

# print(player_stats_2022.dtypes)
col_dtypes = player_stats_2022.dtypes
# [String, String, String, String, String, String, Int32, String, String, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Float64, Float64, Int32, Float64, Int32, Int32, Int32, Int32, Int32, Int32, Float64, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Float64, Int32, Float64, Float64, Float64, Float64, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Float64, Float64, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Float64, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, String, String, String, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Float64, Int32, Int32, Int32, Int32, String, Float64, Float64]


# print(player_stats_2022.describe())

# better format for reading columns
# for col in col_headers:
#     print(col)

# positions and position groups from player data
# print("All positions:", player_stats_2022['position'].unique().to_list())
# print("All groups:", player_stats_2022['position_group'].unique().to_list())

all_positions = ['P', 'TE', 'OT', 'QB', 'FB', 'S', 'C', 'CB', 'DL', 'WR', 'MLB', 'OLB', 'DE', None, 'DB', 'LS', 'NT', 'SAF', 'G', 'DT', 'ILB', 'FS', 'K', 'RB', 'LB']
all_groups = ['DL', None, 'RB', 'TE', 'OL', 'WR', 'QB', 'DB', 'LB', 'SPEC']

fantasy_relevant_positions = ['TE', 'QB', 'WR', 'K', 'RB']


In [22]:
# export some data to excel for closer viewing
fantasy_offense_cols = [
    # Player identification
    'player_id', 'player_display_name', 'position', 'recent_team', 'season',

    # Passing (QB)
    'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'passing_2pt_conversions',

    # Rushing (RB/QB)
    'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles_lost', 'rushing_2pt_conversions',

    # Receiving (WR/RB/TE) - PPR CRITICAL
    'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles_lost',
    'receiving_2pt_conversions',

    # Kicking
    'fg_made', 'fg_missed', 'pat_made', 'fg_made_0_19', 'fg_made_20_29', 'fg_made_30_39', 'fg_made_40_49',
    'fg_made_50_59', 'fg_made_60_',

    # ALREADY CALCULATED (use these directly)
    'fantasy_points', 'fantasy_points_ppr'
]

#note: pat_blocked and fg_blocked record blocks for a kicker's attempts not the defense's blocks of another team

# headers still needed for def	Sacks	Int	Fum Rec	Safeties	Def TD	Return TD	Pts Allowed	PtsAllowedScoring

# export_data = player_stats_2022[fantasy_offense_cols].filter(pl.col('position').is_in(fantasy_relevant_positions))
create_export_data = (
    player_stats_2022[fantasy_offense_cols].lazy()
    .filter(pl.col('position').is_in(fantasy_relevant_positions))
    .sort(['season', 'position'])
)

export_data = create_export_data.collect()

path = pathlib.Path.cwd() / "player_stats_2022.csv"

print(export_data.write_csv(str(path), include_header=True))

None


In [18]:
# get data for team data exploration
team_data_2022 = nfl.load_team_stats(seasons=2022, summary_level="reg")

In [20]:
# exploratory analysis of player data
exploration_data = team_data_2022
print(type(exploration_data))

# col headers

# print(exploration_data.columns)
col_headers = exploration_data.columns
# ['season', 'team', 'season_type', 'games', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'special_teams_tds', 'def_tackles_solo', 'def_tackles_with_assist', 'def_tackle_assists', 'def_tackles_for_loss', 'def_tackles_for_loss_yards', 'def_fumbles_forced', 'def_sacks', 'def_sack_yards', 'def_qb_hits', 'def_interceptions', 'def_interception_yards', 'def_pass_defended', 'def_tds', 'def_fumbles', 'def_safeties', 'misc_yards', 'fumble_recovery_own', 'fumble_recovery_yards_own', 'fumble_recovery_opp', 'fumble_recovery_yards_opp', 'fumble_recovery_tds', 'penalties', 'penalty_yards', 'timeouts', 'punt_returns', 'punt_return_yards', 'kickoff_returns', 'kickoff_return_yards', 'fg_made', 'fg_att', 'fg_missed', 'fg_blocked', 'fg_long', 'fg_pct', 'fg_made_0_19', 'fg_made_20_29', 'fg_made_30_39', 'fg_made_40_49', 'fg_made_50_59', 'fg_made_60_', 'fg_missed_0_19', 'fg_missed_20_29', 'fg_missed_30_39', 'fg_missed_40_49', 'fg_missed_50_59', 'fg_missed_60_', 'fg_made_list', 'fg_missed_list', 'fg_blocked_list', 'fg_made_distance', 'fg_missed_distance', 'fg_blocked_distance', 'pat_made', 'pat_att', 'pat_missed', 'pat_blocked', 'pat_pct', 'gwfg_made', 'gwfg_att', 'gwfg_missed', 'gwfg_blocked', 'gwfg_distance_list']

# col datatypes

# print(exploration_data.dtypes)
col_dtypes = exploration_data.dtypes
# [Int32, String, String, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Float64, Float64, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Float64, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Float64, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Float64, Float64, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Float64, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Int32, String, String, String, Int32, Int32, Int32, Int32, Int32, Int32, Int32, Float64, Int32, Int32, Int32, Int32, String]

print(exploration_data.describe())


# positions and position groups from player data
# print("All positions:", exploration_data['position'].unique().to_list())
# print("All groups:", exploration_data['position_group'].unique().to_list())

<class 'polars.dataframe.frame.DataFrame'>
['season', 'team', 'season_type', 'games', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'special_teams_tds', 'def_tackles_solo', 'def_tackles_with_assist', 'def_tackle_assists', 'def_tackles_for_loss', 'def_tackles_for_loss_yards', 'def_fumbles_forced', 'def_sacks', 'def_sack_yards', 'def_qb_hits', 'def_interceptions', 'def_int

In [21]:
# export some data to excel for closer viewing
fantasy_dst_cols = [
    # IDENTIFICATION
    'season', 'team', 'games',

    # SACKS
    'def_sacks',

    # TURNOVERS
    'def_interceptions', 'def_fumbles_forced', 'fumble_recovery_opp',

    # TDs (Defense/Special Teams)
    'def_tds', 'special_teams_tds', 'fumble_recovery_tds',

    # SAFETIES
    'def_safeties',
]

#note: pat_blocked and fg_blocked record blocks for a team's attempts not the defense's blocks of another team

# headers still needed for def	Sacks	Int	Fum Rec	Safeties	Def TD	Return TD	Pts Allowed	PtsAllowedScoring

# export_data = player_stats_2022[fantasy_offense_cols].filter(pl.col('position').is_in(fantasy_relevant_positions))
create_export_data = (
    team_data_2022[fantasy_dst_cols].lazy()
    .sort(['season'])
)

export_data = create_export_data.collect()

path = "./dst_stats_2022.csv"

print(export_data.write_csv(path, include_header=True))

None
